# EEG-Emotion2Text Kaggle Pipeline (2-Level Text)

当前版本使用两层文本监督:
- L1: Window Level (按情绪，可所有情绪共用同一文本)
- L2: Trial Level (80 个 Trial 视频描述)

并支持 Saveinfo 标签解析、断点续训、训练时长上限。

In [ ]:
# !pip -q install scipy transformers accelerate

In [ ]:
# Kaggle 常用路径初始化 Block
import sys
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

candidate_roots = [
    Path.cwd(),
    KAGGLE_WORKING / 'EEG_emotion2text',
    KAGGLE_INPUT / 'eeg-emotion2text',
    KAGGLE_INPUT / 'eeg-emotion2text' / 'Emotion2text',
]
for p in KAGGLE_INPUT.rglob('eeg2text_core.py'):
    candidate_roots.append(p.parent)

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'eeg2text_core.py').exists() and (root / 'eeg2text_train.py').exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('未找到 eeg2text_core.py / eeg2text_train.py，请确认文件已上传到 Kaggle。')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
from eeg2text_core import CFG
from eeg2text_train import run_loso

In [ ]:
cfg = CFG(
    # 你的 Kaggle 目录结构: {EEG_features, save_info, Emotion2text}
    data_root='/kaggle/input/eeg-emotion2text/EEG_features',
    saveinfo_dir='/kaggle/input/eeg-emotion2text/save_info',
    l1_l2_text_csv_path='/kaggle/input/eeg-emotion2text/Emotion2text/text_protocol_template.csv',
    work_dir='/kaggle/working/eeg2text_ckpt',

    epochs=20,
    batch_size=128,

    l1_weight=0.4,
    l2_weight=0.6,

    same_emotion_weight=1.0,
    pos_neg_margin=0.20,
    margin_loss_weight=0.20,

    resume=True,
    save_every_n_steps=100,
    max_train_hours=8.8,
    time_buffer_minutes=10
)
cfg

In [ ]:
# 如果路径写错，run_loso 会尝试自动发现 EEG_features/save_info/Emotion2text 下的 CSV
results = run_loso(cfg, run_all_folds=False)  # 最终实验改为 True
results

## CSV 协议 (L1 + L2)

文件: `text_protocol_l1_l2.csv`

```csv
emotion,l1_text,trial,l2_text
neutral,Window-level text for neutral emotion.,,
joy,Window-level text for joy emotion.,,
sadness,Window-level text for sadness emotion.,,
fear,Window-level text for fear emotion.,,
disgust,Window-level text for disgust emotion.,,
anger,Window-level text for anger emotion.,,
surprise,Window-level text for surprise emotion.,,
,,1,Trial 1 video description...
,,2,Trial 2 video description...
```

规则:
1. L1: 逐情绪填写（一个情绪一条共享文本）。
2. L2: 填写 `trial=1..80` 对应视频描述。
3. 缺失字段自动回退默认模板。